# PostgreSQL views check

This notebook verifies views in the target schema.

It includes:
- view count
- view list
- definitions
- optional row-count probe (`count(*)`) per view

In [1]:
import os
from pathlib import Path

import psycopg
from psycopg.rows import dict_row


def load_env(path: Path) -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env(Path.cwd().parent / ".env")

DB_CONFIG = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": int(os.getenv("PGPORT", "5432")),
    "dbname": os.getenv("PGDATABASE"),
    "user": os.getenv("PGUSER"),
    "password": os.getenv("PGPASSWORD"),
}
SCHEMA = "s_grnplm_as_t_didsd_nnn_db_tmd"

conn = psycopg.connect(**DB_CONFIG, row_factory=dict_row)
print("Connected")

Connected


In [2]:
# Count views in schema
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT count(*) AS view_count
        FROM information_schema.views
        WHERE table_schema = %s
        """,
        (SCHEMA,),
    )
    view_count = cur.fetchone()["view_count"]

print({"schema": SCHEMA, "view_count": view_count})

{'schema': 's_grnplm_as_t_didsd_nnn_db_tmd', 'view_count': 0}


In [3]:
# List view names
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT table_name AS view_name
        FROM information_schema.views
        WHERE table_schema = %s
        ORDER BY table_name
        """,
        (SCHEMA,),
    )
    views = [r["view_name"] for r in cur.fetchall()]

views

[]

In [4]:
# View definitions (truncated preview)
with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            table_name AS view_name,
            view_definition
        FROM information_schema.views
        WHERE table_schema = %s
        ORDER BY table_name
        """,
        (SCHEMA,),
    )
    defs = cur.fetchall()

for d in defs:
    d["view_definition"] = (d["view_definition"] or "")[:500]

defs

[]

In [5]:
# Row-count probe for each view (safe best-effort)
results = []
with conn.cursor() as cur:
    for view_name in views:
        try:
            sql = f'SELECT count(*) AS cnt FROM "{SCHEMA}"."{view_name}"'
            cur.execute(sql)
            cnt = cur.fetchone()["cnt"]
            results.append({"view": view_name, "count": cnt, "status": "ok"})
        except Exception as exc:
            results.append({"view": view_name, "count": None, "status": str(exc)[:200]})
            conn.rollback()

results

[]

In [6]:
conn.close()
print("Connection closed")

Connection closed
